# Phase 1 bank churn data preparation
Run from the phase1 directory, from top to bottom. This notebook applies the four parts of the repository's Telco walkthrough to the banking file. Each customer has a binary outcome: Exited = 1 if the customer left the bank, and 0 otherwise.

[Published dataset](https://www.kaggle.com/datasets/shrutimechlearn/churn-modelling) · [Cleaning specification](./clean_data.py)

## Part A — Load and inspect
Read the repository CSV as both typed data and literal strings. The string view preserves blanks and placeholders for inspection. Show example predictors without exposing names or identifiers.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

source = Path("data_raw/Churn_Modelling.csv")
bank = pd.read_csv(source)
literal = pd.read_csv(source, dtype=str, keep_default_na=False)
identifiers = ["RowNumber", "CustomerId", "Surname"]
print("Raw shape:", bank.shape)
print(bank.drop(columns=identifiers).head(3).to_string(index=False))
print(bank.dtypes.to_string())

Raw shape: (10000, 14)
 CreditScore Geography Gender  Age  Tenure   Balance  NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  Exited
         619    France Female   42       2      0.00              1          1               1        101348.88       1
         608     Spain Female   41       1  83807.86              1          0               1        112542.58       0
         502    France Female   42       8 159660.80              3          1               0        113931.57       1
RowNumber            int64
CustomerId           int64
Surname             object
CreditScore          int64
Geography           object
Gender              object
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64


## Part B — Evaluate data quality
### Missing values, duplicate records and formats
Test standard nulls, blank or missing-like strings and excess whitespace. Count repeats again after removing identity fields and the outcome so that a unique customer number cannot hide repeated profiles.

In [2]:
features = bank.drop(columns=identifiers + ["Exited"])
trimmed = literal.apply(lambda s: s.str.strip())
missing_tokens = ["", "na", "n/a", "nan", "null", "none", "?"]
print("Parsed missing cells:", bank.isna().sum().sum())
print("Blank or placeholder cells:", trimmed.apply(lambda s: s.str.lower().isin(missing_tokens)).sum().sum())
print("Cells with outer whitespace:", literal.ne(trimmed).sum().sum())
print("Repeated rows:", bank.duplicated().sum())
print("Repeated customer IDs:", bank.CustomerId.duplicated().sum())
print("Repeated predictor profiles:", features.duplicated().sum())
assert bank.RowNumber.tolist() == list(range(1, len(bank) + 1))

Parsed missing cells: 0
Blank or placeholder cells: 0
Cells with outer whitespace: 0
Repeated rows: 0
Repeated customer IDs: 0
Repeated predictor profiles: 0


### Categories and numeric domains
Inspect each categorical or binary field. Check the bank file for numeric conversion failures, nonfinite numbers, negative monetary values and invalid counts before transforming any field.

In [3]:
for col in ["Geography", "Gender", "HasCrCard", "IsActiveMember", "Exited"]:
    print(col, bank[col].value_counts().sort_index().to_dict())
for col in ["HasCrCard", "IsActiveMember", "Exited"]:
    assert set(bank[col].unique()) == {0, 1}
assert set(bank.Gender.unique()) == {"Female", "Male"}
assert set(bank.Geography.unique()) == {"France", "Germany", "Spain"}
numeric = bank.columns.difference(["Geography", "Gender", "Surname"])
parsed = literal[numeric].apply(pd.to_numeric, errors="coerce")
assert np.isfinite(parsed.to_numpy()).all()
assert bank.Age.gt(0).all() and bank.Tenure.ge(0).all()
assert bank.NumOfProducts.ge(1).all()
assert bank[["Balance", "EstimatedSalary", "CreditScore"]].ge(0).all().all()
print("Numeric parsing and domain checks passed.")

Geography {'France': 5014, 'Germany': 2509, 'Spain': 2477}
Gender {'Female': 4543, 'Male': 5457}
HasCrCard {0: 2945, 1: 7055}
IsActiveMember {0: 4849, 1: 5151}
Exited {0: 7963, 1: 2037}
Numeric parsing and domain checks passed.


### Outliers and class imbalance
Use Tukey fences for continuous or ordered predictors. Counts below and above each fence are flags for inspection. A valid zero balance, an older customer or a high product count can carry information.

In [4]:
for col in ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]:
    values = bank[col]
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print(f"{col:16} range {values.min():,.2f} to {values.max():,.2f} | fences {lower:,.2f}, {upper:,.2f} | flagged below {values.lt(lower).sum()}, above {values.gt(upper).sum()}")
print(bank.Exited.value_counts().sort_index().to_string())
print(f"Churn prevalence: {bank.Exited.mean():.2%}")
print(bank.groupby("NumOfProducts").Exited.agg(["size", "sum", "mean"]).to_string())

CreditScore      range 350.00 to 850.00 | fences 383.00, 919.00 | flagged below 15, above 0
Age              range 18.00 to 92.00 | fences 14.00, 62.00 | flagged below 0, above 359
Tenure           range 0.00 to 10.00 | fences -3.00, 13.00 | flagged below 0, above 0
Balance          range 0.00 to 250,898.09 | fences -191,466.36, 319,110.60 | flagged below 0, above 0
NumOfProducts    range 1.00 to 4.00 | fences -0.50, 3.50 | flagged below 0, above 60
EstimatedSalary  range 11.58 to 199,992.48 | fences -96,577.10, 296,967.45 | flagged below 0, above 0
Exited
0    7963
1    2037
Churn prevalence: 20.37%
               size   sum      mean
NumOfProducts                      
1              5084  1409  0.277144
2              4590   348  0.075817
3               266   220  0.827068
4                60    60  1.000000


## Part C — Apply the bank cleaning rules
Drop the row number, customer ID and surname after checking identity. Map the two verified gender labels and rename the target. Preserve the numeric balance and salary, then add the two indicator variables specified by the bank script.

In [5]:
clean = bank.drop(columns=identifiers).copy()
clean["IsMale"] = clean.pop("Gender").map({"Male": 1, "Female": 0})
clean["ZeroBalance"] = clean.Balance.eq(0).astype(int)
clean["NumOfProducts"] = clean.NumOfProducts.clip(upper=3)
clean["LowSalaryFlag"] = clean.EstimatedSalary.lt(1000).astype(int)
clean = clean.rename(columns={"Exited": "Churn"})
print(clean[["IsMale", "ZeroBalance", "LowSalaryFlag"]].sum().to_string())
print(clean.NumOfProducts.value_counts().sort_index().to_string())

IsMale           5457
ZeroBalance      3617
LowSalaryFlag      59
NumOfProducts
1    5084
2    4590
3     326


The cleaned value 3 in NumOfProducts denotes three or more products. The original four-product group has 60 customers. Grouping retains those rows while losing the distinction between three and four. The low-salary threshold is a project rule in the recorded units. It does not establish that any salary is erroneous.

Keep Geography as a categorical predictor. Fit encoding, scaling and any balancing on training folds in Phase 2. An always-stayed prediction scores 79.63% accuracy and misses every customer who left.

## Part D — Validate and save
Check the labels, row order, flags and product grouping against the original measurements. Save the clean bank CSV with no index, read it back and compare all values and types.

In [6]:
assert clean.shape == (len(bank), 13)
assert clean.isna().sum().sum() == 0
assert set(identifiers + ["Gender", "Exited"]).isdisjoint(clean.columns)
pd.testing.assert_series_equal(clean.Churn, bank.Exited, check_names=False)
assert clean.IsMale.eq(bank.Gender.eq("Male").astype(int)).all()
assert clean.ZeroBalance.eq(bank.Balance.eq(0).astype(int)).all()
assert clean.LowSalaryFlag.eq(bank.EstimatedSalary.lt(1000).astype(int)).all()
assert clean.NumOfProducts.eq(bank.NumOfProducts.replace({4: 3})).all()
output = Path("data_clean/bank_clean.csv")
output.parent.mkdir(exist_ok=True)
clean.to_csv(output, index=False)
pd.testing.assert_frame_equal(pd.read_csv(output), clean)
print(f"Saved {output}: {len(clean):,} rows, {len(clean.columns)} columns, {clean.Churn.sum():,} exits.")

Saved data_clean/bank_clean.csv: 10,000 rows, 13 columns, 2,037 exits.


### Compare with the repository script
The original script expects bank.csv, telco.csv and iran.csv. Give it temporary aliases for the three repository inputs. Run the unchanged script in a temporary directory, then check every column, row, value and dtype in its banking output.

In [7]:
import shutil
import subprocess
import sys
import tempfile

aliases = {"Churn_Modelling.csv": "bank.csv", "Telco_Customer_Churn.csv": "telco.csv", "Customer_Churn.csv": "iran.csv"}
with tempfile.TemporaryDirectory() as folder:
    stage = Path(folder)
    for filename, alias in aliases.items():
        shutil.copyfile(Path("data_raw") / filename, stage / alias)
    subprocess.run([sys.executable, "clean_data.py", str(stage), str(stage)], check=True, capture_output=True)
    reference = pd.read_csv(stage / "bank_clean.csv")
pd.testing.assert_frame_equal(pd.read_csv(output), reference)
print("Verified: the notebook reproduces the bank output from clean_data.py.")

Verified: the notebook reproduces the bank output from clean_data.py.
